# Offsets


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
MATCHED_PATH = DATA_DIR / "desi_xray_matched.csv"


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. LOAD DATA
file_path = MATCHED_PATH
df = pd.read_csv(file_path)

# 2. ENSURE NUMERIC TYPE
df["offset_r200c"] = pd.to_numeric(df["offset_r200c"], errors="coerce")

# Keep only rows with valid offsets
df_clean = df.dropna(subset=["offset_r200c"]).copy()

# 3. CALCULATE STATISTICS FOR THRESHOLD (0.1 R200c)
threshold = 0.1
n_total = len(df_clean)
n_relaxed = len(df_clean[df_clean["offset_r200c"] <= threshold])
pct_relaxed = (n_relaxed / n_total) * 100 if n_total > 0 else 0

# 4. PLOTTING
plt.figure(figsize=(10, 6))

plt.hist(
    df_clean["offset_r200c"],
    bins=np.linspace(0, 0.6, 20),
    color="#3274a1",
    alpha=0.7,
    density=True,
    edgecolor="white",
    linewidth=0.5,
    label=f"Total Matched Sample (N={n_total})"
)

plt.axvline(
    threshold,
    color="black",
    linestyle="--",
    linewidth=2,
    label=f"Relaxed Threshold ({threshold} $R_{{200c}}$)\n$N \\leq 0.1$: {n_relaxed} ({pct_relaxed:.1f}%)"
)

# Labels and styling
plt.title("Distribution of X-ray Center to Halo Center Offsets", fontsize=14)
plt.xlabel("Offset / $R_{200c}$", fontsize=12)
plt.ylabel("Normalized Frequency", fontsize=12)

plt.legend(fontsize=11, loc="upper right", frameon=True)
plt.grid(axis="y", alpha=0.2, linestyle="--")
plt.xlim(0, 0.6)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# LOAD DATA
# ==========================================
file_path = MATCHED_PATH
df = pd.read_csv(file_path)

df["offset_r200c"] = pd.to_numeric(
    df["offset_r200c"],
    errors="coerce"
)

df = df.dropna(subset=["offset_r200c"]).copy()

# ==========================================
# SPLIT CATALOGS
# ==========================================
df_bulbul = df[
    df["xray_source"].str.lower() == "bulbul"
].copy()

df_balzer = df[
    df["xray_source"].str.lower() == "balzer"
].copy()

# ==========================================
# SUMMARY STATISTICS
# ==========================================
median_bulbul = df_bulbul["offset_r200c"].median()
median_balzer = df_balzer["offset_r200c"].median()

# ==========================================
# PLOT
# ==========================================
bins = np.linspace(0, 0.6, 25)

plt.figure(figsize=(10, 6))

# Bulbul
plt.hist(
    df_bulbul["offset_r200c"],
    bins=bins,
    density=True,
    histtype="step",
    linewidth=3,
    label=f"Bulbul (N={len(df_bulbul)})"
)

# Balzer
plt.hist(
    df_balzer["offset_r200c"],
    bins=bins,
    density=True,
    histtype="step",
    linewidth=3,
    label=f"Balzer (N={len(df_balzer)})"
)

# Median lines
plt.axvline(
    median_bulbul,
    linestyle="--",
    linewidth=2,
    label=f"Bulbul Median = {median_bulbul:.3f}"
)

plt.axvline(
    median_balzer,
    linestyle=":",
    linewidth=2,
    label=f"Balzer Median = {median_balzer:.3f}"
)

# ==========================================
# LABELS
# ==========================================
plt.title(
    "X-ray Center to Halo Center Offset Distribution: Bulbul vs Balzer",
    fontsize=15
)

plt.xlabel(
    r"Offset / $R_{200c}$",
    fontsize=13
)

plt.ylabel(
    "Probability Density",
    fontsize=13
)

plt.xlim(0, 0.6)

plt.grid(
    axis="y",
    alpha=0.2,
    linestyle="--"
)

plt.legend(
    fontsize=10,
    frameon=True
)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# LOAD DATA
# ==========================================
file_path = MATCHED_PATH
df = pd.read_csv(file_path)

# ==========================================
# CLEAN OFFSET
# ==========================================
df["offset_r200c"] = pd.to_numeric(df["offset_r200c"], errors="coerce")
df = df.dropna(subset=["offset_r200c"]).copy()

# ==========================================
# ENSURE BOOLEAN FORMAT
# ==========================================
df["multiple_xray_candidates"] = df["multiple_xray_candidates"].astype(bool)
df["multiple_halo_candidates"] = df["multiple_halo_candidates"].astype(bool)

# ==========================================
# SPLITS (CORRECTED DEFINITIONS)
# ==========================================

# Clean systems
clean = df[
    ~df["multiple_xray_candidates"] &
    ~df["multiple_halo_candidates"]
]

# TYPE I: multiple X-ray candidates per halo
type1 = df[df["multiple_xray_candidates"]]

# TYPE II: multiple halos per X-ray source
type2 = df[df["multiple_halo_candidates"]]

# ==========================================
# STATS
# ==========================================
def med(x):
    return np.median(x) if len(x) > 0 else np.nan

print("Median offsets:")
print(f"Clean: {med(clean['offset_r200c']):.3f}")
print(f"Type I (multiple X-ray per halo): {med(type1['offset_r200c']):.3f}")
print(f"Type II (multiple halos per X-ray): {med(type2['offset_r200c']):.3f}")

# ==========================================
# PLOT
# ==========================================
bins = np.linspace(0, 0.6, 25)

plt.figure(figsize=(10, 6))

# Clean
plt.hist(
    clean["offset_r200c"],
    bins=bins,
    density=True,
    histtype="step",
    linewidth=3,
    label=f"Clean Matches (N={len(clean)})"
)

# TYPE I
plt.hist(
    type1["offset_r200c"],
    bins=bins,
    density=True,
    histtype="step",
    linewidth=2.5,
    linestyle="--",
    label=f"Type I: Multiple X-ray per halo (N={len(type1)})"
)

# TYPE II
plt.hist(
    type2["offset_r200c"],
    bins=bins,
    density=True,
    histtype="step",
    linewidth=2.5,
    linestyle=":",
    label=f"Type II: Multiple halos per X-ray (N={len(type2)})"
)

# ==========================================
# FORMATTING
# ==========================================
plt.title("Offset Distribution by Ambiguity Type", fontsize=14)
plt.xlabel(r"Offset / $R_{200c}$", fontsize=13)
plt.ylabel("Probability Density", fontsize=13)

plt.xlim(0, 0.6)
plt.grid(alpha=0.2, linestyle="--")

plt.legend(fontsize=10, frameon=True)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import gamma

# ==========================================
# LOAD
# ==========================================
file_path = MATCHED_PATH

df=pd.read_csv(file_path)

# ==========================================
# CLEAN
# ==========================================
df["offset_r200c"]=pd.to_numeric(
    df["offset_r200c"],
    errors="coerce"
)

df=df.dropna(
    subset=["offset_r200c"]
).copy()

offset=df["offset_r200c"].values

print(
    f"N systems = {len(offset):,}"
)

# ==========================================
# FIT GAMMA
# ==========================================

shape,loc,scale=gamma.fit(
    offset,
    floc=0
)

print("\nGamma parameters:")
print(f"shape k = {shape:.3f}")
print(f"scale θ = {scale:.3f}")

mean=shape*scale
sigma=np.sqrt(shape)*scale

print(f"mean = {mean:.3f}")
print(f"sigma = {sigma:.3f}")

# ==========================================
# PLOT
# ==========================================

x=np.linspace(
    0,
    0.8,
    500
)

pdf=gamma.pdf(
    x,
    a=shape,
    loc=0,
    scale=scale
)

plt.figure(figsize=(6,4))

plt.hist(
    offset,
    bins=np.linspace(0,.8,25),
    density=True,
    alpha=.6,
    label=f"Matched sample (N={len(offset)})"
)

plt.plot(
    x,
    pdf,
    linewidth=3,
    label="Gamma fit"
)

plt.xlabel(
    r"Offset / $R_{200c}$"
)

plt.ylabel(
    "Probability Density"
)

plt.legend()

plt.tight_layout()

plt.show()

print(
    f"Median={np.median(offset):.3f}"
)

print(
    f"68 percentile={np.percentile(offset,68):.3f}"
)

print(
    f"90 percentile={np.percentile(offset,90):.3f}"
)
